# Paso 2, antes de entrenar: barrido del learning rate

Busca el **LR más alto con el que el ajuste a la traducción es estable**, para la condición `CONDICION` (`solo_paso2`, `it_solo_paso2` o `paso1+2`). Cada ensayo entrena una LoRA nueva `EPOCAS_ENSAYO` época(s) con calentamiento y coseno completos, la evalúa y la quita (`unload`): el modelo se carga una sola vez.

Un ensayo es **aceptable** si:
1. **la pérdida no explota**: es finita y, pasado el calentamiento, nunca supera `FACTOR_EXPLOSION` veces la pérdida inicial;
2. **aprende**: la pérdida de validación (sobre la respuesta) baja respecto del modelo sin ajustar;
3. **no olvida español general**: la perplejidad de 100 noticias del corpus sube a lo más `UMBRAL_OLVIDO`. Por defecto `UMBRAL_OLVIDO = None`: el olvido se mide y se reporta, pero no decide (con 5 % ningún LR pasaba en Gemma 4 E4B);
4. **(solo instruct) sigue obedeciendo**: de 8 instrucciones generales con verificación automática (`comun.INSTRUCCIONES_PRUEBA`: capital de Francia en una palabra, 7 x 8 solo el número, tres frutas una por línea, responder en inglés, ...), no pierde más de `TOLERANCIA_INSTRUCCIONES` respecto del modelo sin ajustar. Si el ajuste lo deja traduciendo todo al español neutro, falla aquí.

La inestabilidad y el olvido crecen con el LR, así que "aceptable" es (casi) monótono y se puede bisecar en escala log entre `LR_MIN` y `LR_MAX`. El LR más alto aceptable es el **borde**; se **recomienda** el ensayo aceptable con menor pérdida de validación.

Cada ensayo se guarda en el Hub al terminar (`paso2/barrido_lr/<condición>/`); si la sesión se corta, no se repite.

In [ ]:
%pip install -q -U "transformers>=5.5" "peft>=0.17" accelerate datasets huggingface_hub sacrebleu
%pip uninstall -q -y torchao   # el torchao de Colab (0.10) rompe a peft al crear el LoRA; no se usa

In [ ]:
import torch
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
p = torch.cuda.get_device_properties(0)
print(f"GPU: {p.name} | {p.total_memory / 1e9:.0f} GB | bf16: {torch.cuda.is_bf16_supported()}")

In [ ]:
# Configuración
REPO_HF = "mariopedemonte12/chineutro-gemma4"   # repo privado con datos, checkpoints y resultados; None = solo local
DIR_TRABAJO = None   # None = salidas/gemma4 dentro del repo git, o /content/chineutro en Colab
CONDICION = "solo_paso2"   # "solo_paso2" | "paso1+2" | "sin_ajuste" | "solo_paso1" | "it_solo_paso2" | "it_sin_ajuste"
MODELO_BASE = "google/gemma-4-E4B"
MODELO_IT = "google/gemma-4-E4B-it"
NOMBRE_PASO1 = "mlp_r64_5M_priotw80"         # carpeta de DIR_TRABAJO/paso1 con el adaptador a fusionar

MODULOS_PASO2 = ("atencion", "mlp")
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
LR = 2e-4
EPOCAS = 3
LOTE = 16
MAX_LEN = 160
SEED = 42
DTYPE = "bfloat16"                   # "float32" solo para pruebas en GPUs sin bf16 (T4)
MAX_PASOS = -1                       # > 0 corta el entrenamiento en ese paso (pruebas)
# Barrido del LR: búsqueda binaria en escala log del LR más alto con entrenamiento estable
LR_MIN, LR_MAX = 1e-5, 3e-3
ITERACIONES = 4
EPOCAS_ENSAYO = 1                # ~200 pasos con lote 16; cada ensayo tiene su calentamiento y coseno completos
FACTOR_EXPLOSION = 1.5           # la pérdida explota si, pasado el calentamiento, supera 1.5 veces la inicial
UMBRAL_OLVIDO = None             # alza máxima de la perplejidad en noticias; None = solo se reporta, no decide
                                 # (con 0.05 ningún LR pasaba: ajustar a traducir siempre mueve el español general)
TOLERANCIA_INSTRUCCIONES = 1     # instruct: instrucciones de prueba que se pueden dejar de cumplir


### Datos, checkpoints y resultados: repo privado del Hugging Face Hub

Todo vive en el repo privado `REPO_HF`, que es un espejo de `DIR_TRABAJO`: `DIR_TRABAJO/x/y` queda en `x/y` del repo.

| Ruta en el repo | Contenido |
|---|---|
| `datos/` | `comun.py`, `lanzar.py`, los notebooks, `rag_documentos.jsonl`, `dataset_chilenismos_opus.csv`, `split_oficial.csv` |
| `indice_corpus.parquet`, `config_paso1.json`, `muestra_*.parquet`, `presencia_diccionario_*.csv` | salidas de la exploración (00) |
| `paso1/corpus_<M>M.parquet` | corpus del paso 1 ya construido |
| `paso1/<nombre>/last-checkpoint/` | último checkpoint del paso 1 (se reemplaza en cada guardado) |
| `paso1/<nombre>/adaptador/`, `metricas_paso1.json` | adaptador final y métricas |
| `paso2/<condición>/` | adaptador, predicciones y métricas de cada condición |
| `ejecuciones/` | notebook ejecutado y log de cada corrida sin interfaz (`lanzar.py`) |

**Sin interfaz** (runtime de Colab por API): `python datos/lanzar.py <notebook> --repo REPO_HF -p CONDICION paso1+2`. Corre el notebook con papermill y sube el log cada 10 minutos.

**Dentro del repo git** (local) se usan los datos del repo y `DIR_TRABAJO = salidas/gemma4`. Con `REPO_HF = None` no se sube ni se baja nada.

El token de Hugging Face (variable `HF_TOKEN`, `hf auth login` o el secreto `HF_TOKEN` de Colab) necesita **permiso de escritura** para subir resultados, y acceso al corpus restringido.

In [ ]:
# Entorno: token de Hugging Face, comun.py y los datos, y el repo del Hub
import os, sys, json
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules

# Sin interfaz no se puede pedir el token: tiene que estar en HF_TOKEN, en el archivo de `hf auth login`
# o en los secretos de Colab
from huggingface_hub import get_token, snapshot_download
if not get_token() and EN_COLAB:
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        pass
assert get_token(), "Falta el token de Hugging Face (HF_TOKEN o hf auth login)"


def raiz_repo():
    """Raíz del repo git, si el notebook corre dentro de él."""
    actual = Path.cwd().resolve()
    return next((c for c in [actual, *actual.parents] if (c / "models/gemma4/comun.py").is_file()), None)


RAIZ_REPO = raiz_repo()
if DIR_TRABAJO is None:
    DIR_TRABAJO = RAIZ_REPO / "salidas/gemma4" if RAIZ_REPO else "/content/chineutro"
DIR_TRABAJO = Path(DIR_TRABAJO)
DIR_TRABAJO.mkdir(parents=True, exist_ok=True)

if RAIZ_REPO:   # dentro del repo git: datos y código del repo
    RAIZ, DIR_CODIGO = RAIZ_REPO, RAIZ_REPO / "models/gemma4"
else:           # Colab: comun.py y los datos vienen de datos/ en el repo del Hub
    if REPO_HF:
        snapshot_download(REPO_HF, allow_patterns=["datos/*"], local_dir=DIR_TRABAJO)
    RAIZ = DIR_CODIGO = DIR_TRABAJO / "datos"
    assert (DIR_CODIGO / "comun.py").is_file(), f"No encontré comun.py en {DIR_CODIGO}"
sys.path.insert(0, str(DIR_CODIGO))
import comun

HUB = comun.Hub(REPO_HF, DIR_TRABAJO)   # REPO_HF = None: no sube ni baja nada
for relativa in (comun.RUTA_RAG, comun.RUTA_PARES, comun.RUTA_SPLIT):
    try:
        print("✓", comun.ruta_dato(RAIZ, relativa))
    except FileNotFoundError as e:
        print(f"✗ {e}")
print("Datos:", RAIZ, "| salidas:", DIR_TRABAJO, "| Hub:", REPO_HF or "desactivado")

In [ ]:
# Estilo de gráficos: una serie en azul, grilla y ejes discretos
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

AZUL, NARANJA, TINTA, TINTA_2, GRILLA = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({"axes.edgecolor": GRILLA, "axes.labelcolor": TINTA_2, "xtick.color": TINTA_2,
                     "ytick.color": TINTA_2, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": GRILLA, "grid.linewidth": 0.6, "axes.axisbelow": True,
                     "font.size": 10, "axes.titlesize": 11})
pd.set_option("display.max_colwidth", 200)


def barras(serie, titulo, xlabel, log=False, fmt="{:,.0f}"):
    """Una serie por categoría. En escala log se dibujan puntos: el largo de una barra en log engaña."""
    s = serie.sort_values()
    fig, ax = plt.subplots(figsize=(7, 0.4 * len(s) + 1.2))
    if log:
        ax.plot(s.values, s.index.astype(str), "o", color=AZUL, markersize=8)
        ax.set_xscale("log")
        ax.grid(axis="y", visible=True)
    else:
        ax.barh(s.index.astype(str), s.values, color=AZUL, height=0.6)
        ax.grid(axis="y", visible=False)
    for y, v in enumerate(s.values):
        ax.text(v, y, " " + fmt.format(v), va="center", color=TINTA_2, fontsize=9)
    ax.set_title(titulo, loc="left", color=TINTA)
    ax.set_xlabel(xlabel)
    plt.tight_layout()
    plt.show()


def curva_perdida(historial, titulo, ylabel, guardar=None):
    """Pérdida de entrenamiento (línea) y de validación (puntos) desde trainer.state.log_history."""
    hist = pd.DataFrame(historial)
    fig, ax = plt.subplots(figsize=(7, 3.5))
    if "loss" in hist:
        tr = hist.dropna(subset=["loss"])
        ax.plot(tr.step, tr.loss, color=AZUL, linewidth=2, label="entrenamiento")
    if "eval_loss" in hist:
        ev = hist.dropna(subset=["eval_loss"])
        ax.plot(ev.step, ev.eval_loss, "o", color=NARANJA, markersize=8, markeredgecolor="white", label="validación")
    ax.set_xlabel("paso")
    ax.set_ylabel(ylabel)
    ax.set_title(titulo, loc="left", color=TINTA)
    ax.legend(frameon=False)
    plt.tight_layout()
    if guardar:
        fig.savefig(guardar, dpi=150)
    plt.show()

In [ ]:
assert CONDICION in ("solo_paso2", "paso1+2", "sin_ajuste", "solo_paso1", "it_solo_paso2", "it_sin_ajuste"), CONDICION
CHAT = CONDICION.startswith("it_")                 # instruct: plantilla de chat
MODELO = MODELO_IT if CHAT else MODELO_BASE
USA_PASO1 = CONDICION in ("paso1+2", "solo_paso1")
ENTRENA = CONDICION in ("paso1+2", "solo_paso2", "it_solo_paso2")
ADAPTADOR_PASO1 = DIR_TRABAJO / "paso1" / NOMBRE_PASO1 / "adaptador"
if USA_PASO1:
    HUB.bajar(ADAPTADOR_PASO1)
    assert (ADAPTADOR_PASO1 / "adapter_config.json").exists(), f"Falta el adaptador del paso 1 en {ADAPTADOR_PASO1}"

etiqueta = CONDICION if not ENTRENA else f"{CONDICION}__{'+'.join(MODULOS_PASO2)}_r{LORA_R}"
if USA_PASO1:
    etiqueta += f"__p1-{NOMBRE_PASO1}"
DIR_C = DIR_TRABAJO / "paso2" / etiqueta
DIR_C.mkdir(parents=True, exist_ok=True)
print("Condición:", etiqueta, "| modelo:", MODELO, "| plantilla de chat:", CHAT, "| salidas:", DIR_C)

## 1. Datos

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODELO)
pares = comun.cargar_pares(RAIZ)
por_split = {s: pares[pares.split == s].reset_index(drop=True) for s in ("train", "val", "test")}
display(pares.groupby("split").agg(pares=("clave", "size"), expresiones=("clave", "nunique")))

# Split oficial: las expresiones se comparten entre splits a propósito (4 pares en train, 1 en val o test);
# lo que no puede repetirse es un par ni una frase chilena
frases = {s: set(d.frase_chilena.str.strip().str.lower()) for s, d in por_split.items()}
assert not (frases["train"] & frases["val"]) and not (frases["train"] & frases["test"])     and not (frases["val"] & frases["test"]), "fuga entre splits: una frase está en dos splits"
display(por_split["train"][["termino", "frase_chilena", "frase_neutralizada"]].sample(5, random_state=SEED))

In [ ]:
from datasets import Dataset


def a_dataset(df):
    return Dataset.from_list([comun.tokenizar_par(c, n, tok, MAX_LEN, chat=CHAT)
                              for c, n in zip(df.frase_chilena, df.frase_neutralizada)])


ds_train, ds_val = a_dataset(por_split["train"]), a_dataset(por_split["val"])
largos = [len(x) for x in ds_train["input_ids"]]
print(f"train: {len(ds_train)} | val: {len(ds_val)} | tokens por ejemplo: mediana {np.median(largos):.0f}, máx {max(largos)}")
assert max(largos) < MAX_LEN, "algún ejemplo se truncó: sube MAX_LEN"

e = ds_train[0]
print("\n--- Entrada completa ---\n" + tok.decode(e["input_ids"]))
print("--- Parte con pérdida ---\n" + tok.decode([t for t in e["labels"] if t != -100]))

## 2. Modelo sin ajustar: línea base

In [ ]:
import gc, math
from transformers import Trainer, TrainingArguments

assert ENTRENA, f"{CONDICION} no entrena el paso 2: no hay LR que barrer"
modelo = comun.cargar_modelo(MODELO, adaptador_a_fusionar=str(ADAPTADOR_PASO1) if USA_PASO1 else None, dtype=DTYPE)
COLADOR = comun.ColadorPares(tok.pad_token_id)
DIR_B = DIR_TRABAJO / "paso2" / "barrido_lr" / etiqueta
DIR_B.mkdir(parents=True, exist_ok=True)

# Español general para medir olvido: 100 noticias de la muestra de la exploración
HUB.bajar("muestra_5000.parquet")
if (DIR_TRABAJO / "muestra_5000.parquet").exists():
    m_ = pd.read_parquet(DIR_TRABAJO / "muestra_5000.parquet")
    textos_generales = [comun.limpiar_texto(t, 2000) for t in m_[m_.source == "news"].text.sample(100, random_state=SEED)]
else:
    textos_generales = None
    print("Sin muestra_5000.parquet: no se mide el olvido de español general")


def perdida_val(m):
    args = TrainingArguments(output_dir=str(DIR_B / "tmp"), per_device_eval_batch_size=LOTE, bf16=DTYPE == "bfloat16",
                             remove_unused_columns=False, report_to="none")
    return Trainer(model=m, args=args, eval_dataset=ds_val, data_collator=COLADOR).evaluate()["eval_loss"]


def medir(m):
    r = {"val_loss": perdida_val(m)}
    if textos_generales:
        r["ppl_noticias"] = comun.perplejidad(m, tok, textos_generales, max_len=512, bs=8)
    detalle = None
    if CHAT:
        r["instrucciones_ok"], detalle = comun.seguir_instrucciones(m, tok)
    ejemplos = por_split["val"].sample(5, random_state=SEED)
    r["ejemplos"] = [{"chilena": c, "prediccion": p} for c, p in
                     zip(ejemplos.frase_chilena, comun.traducir(m, tok, ejemplos.frase_chilena.tolist(), chat=CHAT))]
    return r, detalle


HUB.bajar(DIR_B)   # retoma un barrido cortado
RUTA_ENSAYOS, RUTA_BASE = DIR_B / "ensayos.json", DIR_B / "base.json"
if RUTA_BASE.exists():
    BASE = json.loads(RUTA_BASE.read_text())
else:
    BASE, det = medir(modelo)
    BASE["instrucciones_detalle"] = det
    RUTA_BASE.write_text(json.dumps(BASE, indent=2, ensure_ascii=False))
ensayos = json.loads(RUTA_ENSAYOS.read_text()) if RUTA_ENSAYOS.exists() else []
print("Línea base:", {k: v for k, v in BASE.items() if k not in ("ejemplos", "instrucciones_detalle")},
      "| ensayos previos:", len(ensayos))
if CHAT:
    for d in BASE["instrucciones_detalle"]:
        print(("✓ " if d["ok"] else "✗ ") + d["instruccion"], "->", d["respuesta"][:80].replace("\n", " | "))

## 3. Ensayos y búsqueda binaria

In [ ]:
def evaluar(r):
    # veredicto con los criterios actuales (también para ensayos reutilizados de una corrida anterior)
    finito = all(math.isfinite(x) for x in (r["loss_inicial"], r["loss_final"], r["loss_max_tras_calentamiento"], r["val_loss"]))
    r["explota"] = bool(not finito or r["loss_max_tras_calentamiento"] > FACTOR_EXPLOSION * r["loss_inicial"])
    r["aprende"] = bool(finito and r["val_loss"] < BASE["val_loss"])
    r["olvida"] = bool(UMBRAL_OLVIDO is not None and r.get("olvido_noticias_%") is not None
                       and r["olvido_noticias_%"] > 100 * UMBRAL_OLVIDO)
    r["rompe_instrucciones"] = bool(CHAT and r["instrucciones_ok"] < BASE["instrucciones_ok"] - TOLERANCIA_INSTRUCCIONES)
    r["aceptable"] = not (r["explota"] or r["olvida"] or r["rompe_instrucciones"]) and r["aprende"]
    return r


def ensayo(lr):
    global modelo
    previo = next((e for e in ensayos if math.isclose(e["lr"], lr, rel_tol=1e-6)), None)
    if previo:
        print(f"LR {lr:.2e}: ya ensayado -> {'aceptable' if evaluar(previo)['aceptable'] else 'NO aceptable'}")
        return previo
    k = len(ensayos) + 1
    torch.manual_seed(SEED)
    m = comun.agregar_lora(modelo, MODULOS_PASO2, LORA_R, LORA_ALPHA, LORA_DROPOUT)
    args = TrainingArguments(
        output_dir=str(DIR_B / "tmp"), per_device_train_batch_size=LOTE, num_train_epochs=EPOCAS_ENSAYO,
        learning_rate=lr, lr_scheduler_type="cosine", warmup_steps=0.05, weight_decay=0.0,
        bf16=DTYPE == "bfloat16", gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
        logging_steps=5, save_strategy="no", eval_strategy="no", remove_unused_columns=False, report_to="none",
        seed=SEED)
    tr = Trainer(model=m, args=args, train_dataset=ds_train, data_collator=COLADOR,
                 callbacks=[comun.callback_progreso(extra={"ensayo": k, "lr": lr})])
    tr.train()
    perdidas = [h["loss"] for h in tr.state.log_history if "loss" in h]
    tras_calentamiento = perdidas[max(1, len(perdidas) // 10):] or perdidas
    r = {"lr": lr, "loss_inicial": perdidas[0], "loss_max_tras_calentamiento": max(tras_calentamiento),
         "loss_final": perdidas[-1]}
    medidas, detalle = medir(m)
    r.update(medidas)
    if detalle:
        r["instrucciones_detalle"] = detalle
    modelo = m.unload()   # quita la LoRA sin fusionarla
    del tr, m
    gc.collect()
    torch.cuda.empty_cache()

    if not all(math.isfinite(x) for x in perdidas):
        r["loss_max_tras_calentamiento"] = float("inf")
    r["olvido_noticias_%"] = (100 * (r["ppl_noticias"] / BASE["ppl_noticias"] - 1)) if "ppl_noticias" in r else None
    evaluar(r)
    ensayos.append(r)
    RUTA_ENSAYOS.write_text(json.dumps(ensayos, indent=2, ensure_ascii=False))
    HUB.subir(DIR_B, ignorar=["tmp/*"], mensaje=f"Barrido LR paso 2 ({etiqueta}): ensayo {k} (lr {lr:.2e})")
    motivos = [n for n in ("explota", "olvida", "rompe_instrucciones") if r[n]] + ([] if r["aprende"] else ["no aprende"])
    print(f"Ensayo {k} | LR {lr:.2e} | val_loss {r['val_loss']:.3f} (base {BASE['val_loss']:.3f}) | "
          f"loss {r['loss_inicial']:.2f} -> {r['loss_final']:.2f} (máx {r['loss_max_tras_calentamiento']:.2f})"
          + (f" | noticias {r['olvido_noticias_%']:+.1f} %" if r["olvido_noticias_%"] is not None else "")
          + (f" | instrucciones {r['instrucciones_ok']}/8" if CHAT else "")
          + f" -> {'aceptable' if r['aceptable'] else 'NO aceptable: ' + ', '.join(motivos)}")
    return r


for e in ensayos:   # ensayos de una corrida anterior: veredicto con los criterios actuales
    evaluar(e)
lo, hi = math.log10(LR_MIN), math.log10(LR_MAX)
LR_BORDE = None
if ensayo(10 ** hi)["aceptable"]:
    LR_BORDE = 10 ** hi
    print("LR_MAX ya es aceptable: sube LR_MAX si quieres explorar más arriba")
else:
    for _ in range(ITERACIONES):
        medio = (lo + hi) / 2
        if ensayo(10 ** medio)["aceptable"]:
            lo, LR_BORDE = medio, 10 ** medio
        else:
            hi = medio
    if LR_BORDE is None:
        if ensayo(10 ** lo)["aceptable"]:
            LR_BORDE = 10 ** lo
        else:
            print("Ni LR_MIN es aceptable: baja LR_MIN o revisa los criterios")

# El borde es el LR más alto estable, no necesariamente el mejor: se comparan también borde/3 y borde/10
if LR_BORDE:
    for divisor in (3, 10):
        if LR_BORDE / divisor >= LR_MIN:
            ensayo(LR_BORDE / divisor)

## 4. Resultado

In [ ]:
tabla = pd.DataFrame(ensayos).sort_values("lr")
cols = ["lr", "val_loss", "loss_inicial", "loss_final", "loss_max_tras_calentamiento", "olvido_noticias_%",
        *(["instrucciones_ok"] if CHAT else []), "explota", "aprende", "olvida", "rompe_instrucciones", "aceptable"]
display(tabla[cols].round(4))

aceptables = tabla[tabla.aceptable]
LR_RECOMENDADO = float(f"{aceptables.sort_values('val_loss').lr.iloc[0]:.1g}") if len(aceptables) else None

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(tabla.lr, tabla.val_loss, "-", color=AZUL, linewidth=2)
ax.scatter(aceptables.lr, aceptables.val_loss, color=AZUL, s=40, zorder=3, label="aceptable")
malos = tabla[~tabla.aceptable]
ax.scatter(malos.lr, malos.val_loss, color=NARANJA, s=40, zorder=3, label="no aceptable")
ax.axhline(BASE["val_loss"], color=TINTA_2, linestyle="--", linewidth=1)
ax.text(tabla.lr.min(), BASE["val_loss"], " sin ajustar", va="bottom", color=TINTA_2, fontsize=9)
ax.set_xscale("log")
ax.set_xlabel("learning rate")
ax.set_ylabel("pérdida de validación")
ax.set_title(f"Barrido del LR del paso 2: {CONDICION}", loc="left", color=TINTA)
ax.legend(frameon=False, fontsize=9)
plt.tight_layout()
fig.savefig(DIR_B / "barrido_lr.png", dpi=150)
plt.show()

resultado = {"condicion": CONDICION, "etiqueta": etiqueta, "lr_recomendado": LR_RECOMENDADO, "lr_borde": LR_BORDE,
             "criterios": {"factor_explosion": FACTOR_EXPLOSION, "umbral_olvido": UMBRAL_OLVIDO,
                           "tolerancia_instrucciones": TOLERANCIA_INSTRUCCIONES, "epocas_ensayo": EPOCAS_ENSAYO},
             "base": BASE, "ensayos": ensayos}
(DIR_B / "resultado.json").write_text(json.dumps(resultado, indent=2, ensure_ascii=False))
HUB.subir(DIR_B, ignorar=["tmp/*"], mensaje=f"Barrido LR paso 2 ({etiqueta}): resultado")
print(f"{CONDICION}: LR borde {LR_BORDE} | LR recomendado {LR_RECOMENDADO}"
      f"  ->  lanzar.py 02_paso2_traduccion.ipynb -p CONDICION {CONDICION} -y LR {LR_RECOMENDADO}")